This is a trial run. I just want to define my main functions here (Gaussian elimination, random circuit generation and evolution, entropy calculation, etc.) and then test it on small systems. Then I will scale things up in another folder. 

2 separate functions for packing and rank calculations

#TWO SEPARATE FUNCTIONS FOR PACKING AND RANK CALCULATION, BOTH OPTIMIZED WITH NUMBA

import numpy as np
from numba import njit
import time

# --- 1. The Matrix Packer ---
@njit
def pack_binary_matrix(binary_matrix):
    """
    Compresses an (N x M) binary matrix into an (N x W) uint64 matrix.
    Every 64 columns of the original matrix become 1 column of 64-bit integers.
    """
    rows, cols = binary_matrix.shape
    # Calculate how many 64-bit words we need per row (ceiling division)
    words = (cols + 63) // 64 
    
    # Initialize the compressed matrix with 64-bit unsigned integers
    packed = np.zeros((rows, words), dtype=np.uint64)
    
    """Iterate through the original binary matrix and set the corresponding bits in the packed matrix. We do this by 
    calculating the word index and bit index for each 1 in the original matrix, then using bitwise OR with a bit mask to set the bit in the packed matrix.
    The bitmask is created by left-shifting 1 by the bit index by an amount equal to the bit index, and we ensure that the shift is done on a 64-bit type to prevent overflow.
    The bitmask is then ORed with the current value in the packed matrix at the appropriate word index to set the bit corresponding to the 1 in the original matrix.
    We only process the 1s to save time, as the 0s are already"""
    for r in range(rows):
        for c in range(cols):
            if binary_matrix[r, c] == 1: 
                word_idx = c // 64 #extract the word index
                bit_idx = c % 64 #extract the bit index within that word

                # Force the 1 to be a 64-bit type before shifting to prevent overflow. 
                bit_mask = np.uint64(1) << np.uint64(bit_idx)
                packed[r, word_idx] |= bit_mask #manually set the bit using bitwise OR for each 1 in the original matrix
                
    return packed

# --- 2. The Ultra-Fast Rank Solver ---
@njit
def gf2_rank(packed_matrix, num_cols):
    """
    Computes the GF(2) rank directly on the packed uint64 memory blocks.
    Utilizes the upper-triangular shortcut to cut mathematical operations in half.
    """
    mat = packed_matrix.copy()
    rows, words = mat.shape
    rank = 0
    
    for col in range(num_cols):
        # Locate the exact integer and the exact bit within it for the current column
        word_idx = col // 64
        bit_idx = col % 64
        
        # 1. Pivot search (Optimized: Only search from the current rank downwards)
        pivot_row = -1
        for r in range(rank, rows):
            # Shift the target bit to the 0th position and mask it with 1
            if (mat[r, word_idx] >> np.uint64(bit_idx)) & np.uint64(1):
                pivot_row = r
                break
                
        if pivot_row == -1:
            continue
            
        # 2. Row swap
        if pivot_row != rank:
            for w in range(words):
                temp = mat[rank, w]
                mat[rank, w] = mat[pivot_row, w]
                mat[pivot_row, w] = temp
                
        # 3. Row elimination via XOR 
        # (Optimized: Only eliminate strictly BELOW the pivot to form Row Echelon Form)
        for r in range(rank + 1, rows):
            if (mat[r, word_idx] >> np.uint64(bit_idx)) & np.uint64(1):
                # The Hardware Magic: XORing a single word processes 64 bits simultaneously
                for w in range(words):
                    mat[r, w] ^= mat[rank, w]
                    
        rank += 1
        if rank == rows:
            break
            
    return rank

# ==========================================
# BENCHMARK TEST (10000 x 10000)
# ==========================================
if __name__ == "__main__":
    N = 10000
    print(f"Generating random {N} x {N} binary matrix...")
    # Use uint8 for the initial generation to save RAM before packing
    binary_matrix = np.random.randint(0, 2, (N, 2 * N), dtype=np.uint8)
    
    # Pack the matrix
    t0 = time.time()
    packed_mat = pack_binary_matrix(binary_matrix)
    rank = gf2_rank(packed_mat, num_cols=2 *N)
    t1 = time.time()
    
    print(f"Rank: {rank}")
    print(f"Rank calculation time: {t1 - t0:.4f} seconds.")

One single function for both packing and rank calculation, uses matrix packing and passes the output (packed matrix) to the rank calculator to calculate the rank. As of March 23, this is faster.

In [132]:
#ONE SINGLE FUNCTION FOR BOTH PACKING AND RANK CALCULATION, OPTIMIZED WITH NUMBA. THIS DOES NOT PRESERVE THE ORIGINAL MATRIX!!!!!!!!!!!!!

import numpy as np
from numba import njit, test
import time

# --- 1. The Matrix Packer ---
@njit
def rank_binary_matrix(binary_matrix):
    """
    Compresses an (N x M) binary matrix into an (N x W) uint64 matrix.
    Every 64 columns of the original matrix become 1 column of 64-bit integers.
    """
    rows, cols = binary_matrix.shape
    # Calculate how many 64-bit words we need per row (ceiling division)
    words = (cols + 63) // 64 
    
    # Initialize the compressed matrix with 64-bit unsigned integers
    packed = np.zeros((rows, words), dtype=np.uint64)
    
    """Iterate through the original binary matrix and set the corresponding bits in the packed matrix. We do this by 
    calculating the word index and bit index for each 1 in the original matrix, then using bitwise OR with a bit mask to set the bit in the packed matrix.
    The bitmask is created by left-shifting 1 by the bit index by an amount equal to the bit index, and we ensure that the shift is done on a 64-bit type to prevent overflow.
    The bitmask is then ORed with the current value in the packed matrix at the appropriate word index to set the bit corresponding to the 1 in the original matrix.
    We only process the 1s to save time, as the 0s are already"""
    for r in range(rows):
        for c in range(cols):
            if binary_matrix[r, c] == 1: 
                word_idx = c // 64 #extract the word index
                bit_idx = c % 64 #extract the bit index within that word

                # Force the 1 to be a 64-bit type before shifting to prevent overflow. 
                bit_mask = np.uint64(1) << np.uint64(bit_idx)
                packed[r, word_idx] |= bit_mask #manually set the bit using bitwise OR for each 1 in the original matrix
                


    """
    Computes the GF(2) rank directly on the packed uint64 memory blocks.
    Utilizes the upper-triangular shortcut to cut mathematical operations in half.
    """
    rows, words = packed.shape
    rank = 0
    
    for col in range(cols):
        # Locate the exact integer and the exact bit within it for the current column
        word_idx = col // 64
        bit_idx = col % 64
        
        # 1. Pivot search (Optimized: Only search from the current rank downwards)
        pivot_row = -1
        for r in range(rank, rows):
            # Shift the target bit to the 0th position and mask it with 1
            if (packed[r, word_idx] >> np.uint64(bit_idx)) & np.uint64(1):
                pivot_row = r
                break
                
        if pivot_row == -1:
            continue
            
        # 2. Row swap
        if pivot_row != rank:
            for w in range(words):
                temp = packed[rank, w]
                packed[rank, w] = packed[pivot_row, w]
                packed[pivot_row, w] = temp
                
        # 3. Row elimination via XOR 
        # (Optimized: Only eliminate strictly BELOW the pivot to form Row Echelon Form)
        for r in range(rank + 1, rows):
            if (packed[r, word_idx] >> np.uint64(bit_idx)) & np.uint64(1):
                # The Hardware Magic: XORing a single word processes 64 bits simultaneously
                for w in range(words):
                    packed[r, w] ^= packed[rank, w]
                    
        rank += 1
        if rank == rows:
            break
            
    return rank

## ==========================================
# TESTING
## ==========================================
test1 = np.array([[1, 0],
                 [0, 1]])
print("\nTest Matrix 1:")
print(test1 )
print("Calculated Rank:", rank_binary_matrix(test1))

test2 = np.array([[1, 0, 1],
                 [0, 1, 1], 
                   [1, 1, 0]])
print("\nTest Matrix 2:")
print(test2 )
print("Calculated Rank:", rank_binary_matrix(test2))

test3 = np.array([[1, 0],
                 [1, 0], 
                [0, 0]])
print("\nTest Matrix 3:")
print(test3 )
print("Calculated Rank:", rank_binary_matrix(test3))

test4 = np.array([[1, 1, 0, 0],
                 [0, 1, 1, 0],
                 [0, 0, 1, 1],
                 [1, 0, 0, 1]])
print("\nTest Matrix 4:")
print(test4 )
print("Calculated Rank:", rank_binary_matrix(test4))

test5 = np.array([[1]])
print("\nTest Matrix 5:")
print(test5 )
print("Calculated Rank:", rank_binary_matrix(test5))

test6 = np.zeros((10, 10))
print("\nTest Matrix 6 (10x10 zeros):")
print(test6 )
print("Calculated Rank:", rank_binary_matrix(test6))

test7 = np.ones((7, 9))
print("\nTest Matrix 7 (7x9 ones):")
print(test7 )
print("Calculated Rank:", rank_binary_matrix(test7))
    




Test Matrix 1:
[[1 0]
 [0 1]]
Calculated Rank: 2

Test Matrix 2:
[[1 0 1]
 [0 1 1]
 [1 1 0]]
Calculated Rank: 2

Test Matrix 3:
[[1 0]
 [1 0]
 [0 0]]
Calculated Rank: 1

Test Matrix 4:
[[1 1 0 0]
 [0 1 1 0]
 [0 0 1 1]
 [1 0 0 1]]
Calculated Rank: 3

Test Matrix 5:
[[1]]
Calculated Rank: 1

Test Matrix 6 (10x10 zeros):
[[0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]]
Calculated Rank: 0

Test Matrix 7 (7x9 ones):
[[1. 1. 1. 1. 1. 1. 1. 1. 1.]
 [1. 1. 1. 1. 1. 1. 1. 1. 1.]
 [1. 1. 1. 1. 1. 1. 1. 1. 1.]
 [1. 1. 1. 1. 1. 1. 1. 1. 1.]
 [1. 1. 1. 1. 1. 1. 1. 1. 1.]
 [1. 1. 1. 1. 1. 1. 1. 1. 1.]
 [1. 1. 1. 1. 1. 1. 1. 1. 1.]]
Calculated Rank: 1


Entropy calculator (simple)

In [133]:
import numpy as np

def calculate_entropy(stabilizer, n_cut):
    """
    Calculates the entanglement entropy for a given stabilizer matrix and cut.
    """
    L = stabilizer.shape[1] // 2
    print(f"Stabilizer shape: {stabilizer.shape}, L: {L}, n_cut: {n_cut}")  
    
    # 1. Grab X columns for qubits 0 to n-1
    # 2. Grab Z columns for qubits 0 to n-1 (offset by L)
    # We use np.hstack to join them into an L x 2n matrix
    sub_A = np.hstack([stabilizer[:, :n_cut], stabilizer[:, L : L+n_cut]])
    print(f"Submatrix: \n {sub_A} \n")

    rank = rank_binary_matrix(sub_A)
    print(f"Rank: {rank}")

    return rank - n_cut



Simple tests with a small number of qubits to check if code works properly

In [134]:
import numpy as np
from numba import njit
import stim


# 1. Create the 'Instruction' set
circuit = stim.Circuit()
circuit.append("H", [0])
circuit.append("CNOT", [0, 1])


# 2. Create the 'Simulator' (The thing that holds the state)
sim = stim.TableauSimulator()

# 3. Feed the instructions to the simulator
sim.do(circuit)
hasattr(sim, "current_tableau")
# 4. Peek at the Binary Check Matrix (Tableau)
# We use current_inverse_tableau because it gives the 'stabilizers' directly
# Pull the internal Heisenberg state and invert it to get the Schrödinger state
tableau = sim.current_inverse_tableau() ** -1

# 1. to_numpy() returns 6 blocks: 
# (X-to-X, X-to-Z, Z-to-X, Z-to-Z, x_signs, z_signs)
x2x, x2z, z2x, z2z, x_signs, z_signs = tableau.to_numpy()

# 2. Extract the Stabilizers (Z-generators). 
# We horizontally stack the X-components and Z-components.
stabilizer_matrix = np.hstack([z2x, z2z])

# Convert boolean (True/False) to integers (1/0) for your GF(2) solver
stabilizer_matrix = stabilizer_matrix.astype(int)
print(f"Stabilizer Matrix:\n{stabilizer_matrix} \n")

print(f"Entanglement Entropy: {calculate_entropy(stabilizer_matrix, n_cut=1)}")


Stabilizer Matrix:
[[1 1 0 0]
 [0 0 1 1]] 

Stabilizer shape: (2, 4), L: 2, n_cut: 1
Submatrix: 
 [[1 0]
 [0 1]] 

Rank: 2
Entanglement Entropy: 1


In [ ]:
import numpy as np
from numba import njit
import stim


# 1. Create the 'Instruction' set
circuit = stim.Circuit()
circuit.append("H", [0])
circuit.append("CNOT", [0, 1])
circuit.append("CNOT", [0, 2])
circuit.append("CNOT", [0, 3])


# 2. Create the 'Simulator' (The thing that holds the state)
sim = stim.TableauSimulator()

# 3. Feed the instructions to the simulator
sim.do(circuit)
hasattr(sim, "current_tableau")
# 4. Peek at the Binary Check Matrix (Tableau)
# We use current_inverse_tableau because it gives the 'stabilizers' directly
# Pull the internal Heisenberg state and invert it to get the Schrödinger state
tableau = sim.current_inverse_tableau() ** -1

# 1. to_numpy() returns 6 blocks: 
# (X-to-X, X-to-Z, Z-to-X, Z-to-Z, x_signs, z_signs)
x2x, x2z, z2x, z2z, x_signs, z_signs = tableau.to_numpy()

# 2. Extract the Stabilizers (Z-generators). 
# We horizontally stack the X-components and Z-components.
stabilizer_matrix = np.hstack([z2x, z2z])

# Convert boolean (True/False) to integers (1/0) for your GF(2) solver
stabilizer_matrix = stabilizer_matrix.astype(int)
print(f"Stabilizer Matrix:\n{stabilizer_matrix} \n")

print(f"Entanglement Entropy: {calculate_entropy(stabilizer_matrix, n_cut=2)}")


Stabilizer Matrix:
[[1 1 1 1 0 0 0 0]
 [0 0 0 0 1 1 0 0]
 [0 0 0 0 1 0 1 0]
 [0 0 0 0 1 0 0 1]] 

Stabilizer shape: (4, 8), L: 4, n_cut: 2
Submatrix: 
 [[1 1 0 0]
 [0 0 1 1]
 [0 0 1 0]
 [0 0 1 0]] 

Rank: 3
Entanglement Entropy: 1


In [153]:
import numpy as np
from numba import njit
import stim


# 1. Create the 'Instruction' set
circuit = stim.Circuit()
circuit.append("H", [0])
circuit.append("H", [1])
circuit.append("H", [2])
circuit.append("CNOT", [0, 3])
circuit.append("CNOT", [1, 4])
circuit.append("CNOT", [2, 5])


# 2. Create the 'Simulator' (The thing that holds the state)
sim = stim.TableauSimulator()

# 3. Feed the instructions to the simulator
sim.do(circuit)
hasattr(sim, "current_tableau")
# 4. Peek at the Binary Check Matrix (Tableau)
# We use current_inverse_tableau because it gives the 'stabilizers' directly
# Pull the internal Heisenberg state and invert it to get the Schrödinger state
tableau = sim.current_inverse_tableau() ** -1

# 1. to_numpy() returns 6 blocks: 
# (X-to-X, X-to-Z, Z-to-X, Z-to-Z, x_signs, z_signs)
x2x, x2z, z2x, z2z, x_signs, z_signs = tableau.to_numpy()

# 2. Extract the Stabilizers (Z-generators). 
# We horizontally stack the X-components and Z-components.
stabilizer_matrix = np.hstack([z2x, z2z])

# Convert boolean (True/False) to integers (1/0) for your GF(2) solver
stabilizer_matrix = stabilizer_matrix.astype(int)
print(f"Stabilizer Matrix:\n{stabilizer_matrix} \n")

print(f"Entanglement Entropy: {calculate_entropy(stabilizer_matrix, n_cut=3)}")


Stabilizer Matrix:
[[1 0 0 1 0 0 0 0 0 0 0 0]
 [0 1 0 0 1 0 0 0 0 0 0 0]
 [0 0 1 0 0 1 0 0 0 0 0 0]
 [0 0 0 0 0 0 1 0 0 1 0 0]
 [0 0 0 0 0 0 0 1 0 0 1 0]
 [0 0 0 0 0 0 0 0 1 0 0 1]] 

Stabilizer shape: (6, 12), L: 6, n_cut: 3
Submatrix: 
 [[1 0 0 0 0 0]
 [0 1 0 0 0 0]
 [0 0 1 0 0 0]
 [0 0 0 1 0 0]
 [0 0 0 0 1 0]
 [0 0 0 0 0 1]] 

Rank: 6
Entanglement Entropy: 3


Let's start with a simple system.